In [ ]:
import csv
import struct
import threading
import time
from dataclasses import dataclass
from pathlib import Path
from queue import Queue, Empty

import serial
import pandas as pd
import matplotlib.pyplot as plt

from protocol import (
    PROTOCOL_VER,
    FLAG_DATA,
    FLAG_STREAM,
    ID_PC,
    ID_DRONE,
    OPT_TELEMETRY,
    OPT_IDENTI_MOMENT,
    OPT_CAL_PARAM,
    build_frame,
    parse_frame,
)


In [ ]:
@dataclass
class TestConfig:
    # Drone serial port
    drone_port: str = "COM5"  # change
    drone_baud: int = 9600

    # - Za omega_dot: low-pass filter pred odvajanjem zaradi šuma
    # from scipy.signal import savgol_filter
    # omega_filtered = savgol_filter(omega, window_length=11, polyorder=3, axis=0)
    # - ali -
    # omega_dot = savgol_filter(
    #     omega,
    #     window_length=11,
    #     polyorder=3,
    #     deriv=1,
    #     delta=dt,
    #     axis=0
    # )

    # Command schedule 
    # All times below are from test start (seconds)
    command_times_s: tuple[float, ...] = (
        0.0,   # EDF 0, 0deg
        2.0,   # EDF step to selected level, 0deg
        8.5,   # +5
        10.5,  # 0
        12.5,  # -5
        14.5,  # 0
        16.5,  # +10
        18.5,  # 0
        20.5,  # -10
        22.5,  # 0
        24.5,  # +15
        26.5,  # 0
        28.5,  # -15
        30.5,  # 0
        32.5,  # +20
        34.5,  # 0
        36.5,  # -20
        38.5,  # 0
        40.5,  # EDF back to 0, 0deg
    )

    # EDF command per step (%)
    edf_percents: tuple[int, ...] = (
        0,
        40,
        40, 40, 40, 40,
        40, 40, 40, 40,
        40, 40, 40, 40,
        40, 40, 40, 40,
        0,
    )

    # Servo command angles (degrees) for each step
    servo_xp_deg: tuple[int, ...] = (
        0,
        0,
        +5, 0, -5, 0,
        +10, 0, -10, 0,
        +15, 0, -15, 0,
        +20, 0, -20, 0,
        0,
    )
    servo_xn_deg: tuple[int, ...] = servo_xp_deg
    servo_yp_deg: tuple[int, ...] = servo_xp_deg
    servo_yn_deg: tuple[int, ...] = servo_xp_deg

    # Logging
    output_csv: str = "Meritve/moment_test_telemetry.csv"

    # Optional idle time after enabling ident-mode (before schedule starts)
    pre_test_idle_s: float = 1.0

    # How long to keep logging after the last scheduled command
    post_test_wait_s: float = 2.0

cfg = TestConfig()
cfg


In [ ]:
ANGLE_SCALE = 1000.0  # IMPORTANT: set to the same ANGLE_SCALE used in drone firmware


def _s16_from_be(hi: int, lo: int) -> int:
    v = (hi << 8) | lo
    return v - 0x10000 if v & 0x8000 else v


def decode_opt_telemetry(payload: bytes) -> dict | None:
    # Payload layout from drone firmware snippet (big-endian per field):
    # tick_ms:u32, edf_percent:u8,
    # servo_xp/xn/yp/yn:int16 (value * ANGLE_SCALE)
    # roll/pitch/yaw:int16 (value * ANGLE_SCALE)
    # acc_x/y/z:int16 (value * ANGLE_SCALE)
    # gyro_x/y/z:int16 (value * ANGLE_SCALE)
    if len(payload) < 31:
        return None

    tick_ms = (payload[0] << 24) | (payload[1] << 16) | (payload[2] << 8) | payload[3]
    edf_percent = int(payload[4])
    off = 5

    def take_s16() -> int:
        nonlocal off
        v = _s16_from_be(payload[off], payload[off + 1])
        off += 2
        return v

    servo_xp_raw = take_s16(); servo_xn_raw = take_s16(); servo_yp_raw = take_s16(); servo_yn_raw = take_s16()
    roll_raw = take_s16(); pitch_raw = take_s16(); yaw_raw = take_s16()
    acc_x_raw = take_s16(); acc_y_raw = take_s16(); acc_z_raw = take_s16()
    gyro_x_raw = take_s16(); gyro_y_raw = take_s16(); gyro_z_raw = take_s16()

    s = float(ANGLE_SCALE)

    return {
        "tick_ms": tick_ms,
        "edf_percent": edf_percent,

        "servo_xp": servo_xp_raw / s,
        "servo_xn": servo_xn_raw / s,
        "servo_yp": servo_yp_raw / s,
        "servo_yn": servo_yn_raw / s,

        "roll": roll_raw / s,
        "pitch": pitch_raw / s,
        "yaw": yaw_raw / s,

        "acc_x": acc_x_raw / s,
        "acc_y": acc_y_raw / s,
        "acc_z": acc_z_raw / s,

        "gyro_x": gyro_x_raw / s,
        "gyro_y": gyro_y_raw / s,
        "gyro_z": gyro_z_raw / s,
    }


In [ ]:
def build_sequence(cfg: TestConfig):
    times: list[float] = []
    edf: list[int] = []
    sxp: list[int] = []
    sxn: list[int] = []
    syp: list[int] = []
    syn: list[int] = []
    labels: list[str] = []

    def add(dt_s: float, edf_p: int, fin_deg: int, label: str):
        t = (times[-1] if times else 0.0) + float(dt_s)
        times.append(t)
        edf.append(int(edf_p))

        if cfg.apply_fin_to_all_servos:
            sxp.append(int(fin_deg)); sxn.append(int(fin_deg)); syp.append(int(fin_deg)); syn.append(int(fin_deg))
        else:
            # If you want a different mapping, set apply_fin_to_all_servos=True or edit here.
            sxp.append(int(fin_deg)); sxn.append(int(fin_deg)); syp.append(int(fin_deg)); syn.append(int(fin_deg))

        labels.append(str(label))

    # Sequence (requested):
    # 1) EDF=0, fin=0, log 2 s
    add(cfg.log_idle_edf0_s, 0, 0, "EDF0_FIN0_LOG")

    # 2) EDF=chosen level, fin=0, wait 2?3 s
    add(cfg.wait_after_edf_step_s, cfg.edf_level_percent, 0, "EDF_STEP_UP_WAIT")

    # 3) baseline fin=0 log >= 2 s
    add(cfg.baseline_fin0_log_s, cfg.edf_level_percent, 0, "BASELINE_FIN0")

    # 4-8) for each angle: +A (2s), 0 (2s), -A (2s), 0 (2s)
    for a in cfg.fin_angles_deg:
        add(cfg.step_hold_s, cfg.edf_level_percent, +int(a), f"FIN_PLUS_{int(a)}")
        add(cfg.step_hold_s, cfg.edf_level_percent, 0, f"FIN_ZERO_AFTER_PLUS_{int(a)}")
        add(cfg.step_hold_s, cfg.edf_level_percent, -int(a), f"FIN_MINUS_{int(a)}")
        add(cfg.step_hold_s, cfg.edf_level_percent, 0, f"FIN_ZERO_AFTER_MINUS_{int(a)}")

    # 9) EDF back to 0
    add(0.0, 0, 0, "EDF_BACK_TO_0")

    # Convert cumulative times to command_times_s (relative from test start)
    command_times_s = [0.0] + times[:-1]  # command at t=0 plus before each next segment starts
    edf_percents = [0] + edf[:-1]
    sxp_deg = [0] + sxp[:-1]
    sxn_deg = [0] + sxn[:-1]
    syp_deg = [0] + syp[:-1]
    syn_deg = [0] + syn[:-1]
    cmd_labels = ["START"] + labels[:-1]

    return command_times_s, edf_percents, sxp_deg, sxn_deg, syp_deg, syn_deg, cmd_labels


def run_moment_test(cfg: TestConfig) -> Path:
    command_times_s, edf_percents, sxp, sxn, syp, syn, cmd_labels = build_sequence(cfg)

    n = len(command_times_s)
    output_path = Path(cfg.output_csv)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    telemetry_q: Queue = Queue()

    print("Opening drone serial...")
    drone_ser = serial.Serial(cfg.drone_port, cfg.drone_baud, timeout=0.05, dsrdtr=False, rtscts=False)
    time.sleep(0.3)

    reader = DroneTelemetryReader(drone_ser, telemetry_q)
    reader.start()

    last_cmd_edf: int | None = None
    last_cmd_pc_time_s: float | None = None
    last_cmd_step_idx: int | None = None
    last_cmd_label: str | None = None

    fieldnames = [
        "pc_time_s",
        "tick_ms",
        "edf_percent",
        "cmd_step_idx",
        "cmd_label",
        "cmd_edf_percent",
        "cmd_servo_xp_deg",
        "cmd_servo_xn_deg",
        "cmd_servo_yp_deg",
        "cmd_servo_yn_deg",
        "cmd_pc_time_s",
        "servo_xp",
        "servo_xn",
        "servo_yp",
        "servo_yn",
        "roll",
        "pitch",
        "yaw",
        "acc_x",
        "acc_y",
        "acc_z",
        "gyro_x",
        "gyro_y",
        "gyro_z",
    ]

    def drain_and_log(writer: csv.DictWriter) -> int:
        nonlocal last_cmd_edf, last_cmd_pc_time_s, last_cmd_step_idx, last_cmd_label
        wrote = 0
        while True:
            try:
                msg = telemetry_q.get_nowait()
            except Empty:
                break
            msg["cmd_step_idx"] = last_cmd_step_idx if last_cmd_step_idx is not None else ""
            msg["cmd_label"] = last_cmd_label if last_cmd_label is not None else ""
            msg["cmd_edf_percent"] = last_cmd_edf if last_cmd_edf is not None else ""
            msg["cmd_pc_time_s"] = last_cmd_pc_time_s if last_cmd_pc_time_s is not None else ""

            if last_cmd_step_idx is None:
                msg["cmd_servo_xp_deg"] = ""; msg["cmd_servo_xn_deg"] = ""; msg["cmd_servo_yp_deg"] = ""; msg["cmd_servo_yn_deg"] = ""
            else:
                i = int(last_cmd_step_idx)
                msg["cmd_servo_xp_deg"] = int(sxp[i])
                msg["cmd_servo_xn_deg"] = int(sxn[i])
                msg["cmd_servo_yp_deg"] = int(syp[i])
                msg["cmd_servo_yn_deg"] = int(syn[i])

            writer.writerow(msg)
            wrote += 1
        return wrote

    try:
        with open(output_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
            writer.writeheader()

            print("Sending OPT_IDENTI_MOMENT (plen=0)...")
            drone_ser.write(build_ident_moment_frame())
            drone_ser.flush()

            if cfg.pre_test_idle_s > 0:
                t0 = time.monotonic()
                while (time.monotonic() - t0) < cfg.pre_test_idle_s:
                    drain_and_log(writer)
                    time.sleep(0.01)

            print("Generated command schedule:")
            for i in range(n):
                print(
                    f"  step={i:02d} | t={float(command_times_s[i]):7.3f}s | EDF={int(edf_percents[i]):3d}%"
                    f" | fin(deg) xp={int(sxp[i])} xn={int(sxn[i])} yp={int(syp[i])} yn={int(syn[i])} | {cmd_labels[i]}"
                )

            start = time.monotonic()
            for i in range(n):
                due = start + float(command_times_s[i])
                while time.monotonic() < due:
                    drain_and_log(writer)
                    time.sleep(0.005)

                frame = build_calib_frame(
                    edf_pwr_percent=int(edf_percents[i]),
                    x_plus_angle_deg=int(sxp[i]),
                    x_minus_angle_deg=int(sxn[i]),
                    y_plus_angle_deg=int(syp[i]),
                    y_minus_angle_deg=int(syn[i]),
                )
                drone_ser.write(frame)
                drone_ser.flush()

                last_cmd_step_idx = i
                last_cmd_label = str(cmd_labels[i])
                last_cmd_edf = int(edf_percents[i])
                last_cmd_pc_time_s = time.time()

                drain_and_log(writer)

            if cfg.post_test_wait_s > 0:
                print(f"Post-test wait/log: {cfg.post_test_wait_s:.2f}s")
                t0 = time.monotonic()
                while (time.monotonic() - t0) < cfg.post_test_wait_s:
                    drain_and_log(writer)
                    time.sleep(0.01)

    finally:
        try:
            reader.stop()
        except Exception:
            pass
        time.sleep(0.05)
        try:
            drone_ser.close()
        except Exception:
            pass

    print(f"Saved telemetry CSV to {output_path}")
    return output_path


out_csv = run_moment_test(cfg)
out_csv


In [ ]:
# Plot relevant data from the last run (sequence-based schedule)

import numpy as np

df = pd.read_csv(out_csv)

# Time base
if "tick_ms" in df.columns and df["tick_ms"].notna().any():
    t = (df["tick_ms"] - df["tick_ms"].iloc[0]) / 1000.0
    t_label = "drone tick (s)"
else:
    t = df["pc_time_s"] - df["pc_time_s"].iloc[0]
    t_label = "pc time (s)"

df = df.copy()
df["t_s"] = t

def _ffill(series):
    return series.fillna(method="ffill") if hasattr(series, "fillna") else series

# omega_body from drone (use decoded gyro as p,q,r)
df["p"] = df["gyro_x"]
df["q"] = df["gyro_y"]
df["r"] = df["gyro_z"]

# omega_dot = d(omega_body)/dt
# Use numpy.gradient for uneven sampling (requires monotonic t)
t_s = df["t_s"].to_numpy(dtype=float)
for axis in ("p", "q", "r"):
    w = df[axis].to_numpy(dtype=float)
    df[f"{axis}_dot"] = np.gradient(w, t_s)

# Command change markers (when cmd_step_idx changes)
step_idx = _ffill(df["cmd_step_idx"]).fillna(-1)
step_change_idx = df.index[(step_idx.diff().fillna(0) != 0)].tolist()
change_times = df.loc[step_change_idx, "t_s"].tolist()

info = (
    f"steps={len(cfg.command_times_s)} | ANGLE_SCALE={ANGLE_SCALE:g} | pre_idle={cfg.pre_test_idle_s:.2f}s | post_wait={cfg.post_test_wait_s:.2f}s
"
    f"edf_percents={list(cfg.edf_percents)}"
)

fig, axs = plt.subplots(5, 1, figsize=(14, 15), sharex=True)

# 1) EDF: command vs telemetry
axs[0].step(df["t_s"], df["cmd_edf_percent"], where="post", label="EDF cmd (%)", linewidth=2)
axs[0].plot(df["t_s"], df["edf_percent"], label="EDF telemetry (%)", alpha=0.85)
axs[0].set_ylabel("EDF (%)")
axs[0].set_title("Moment test: EDF command vs telemetry")
axs[0].grid(True)
axs[0].legend(loc="best")
axs[0].text(0.01, 0.98, info, transform=axs[0].transAxes, va="top", ha="left", fontsize=9,
            bbox=dict(facecolor="white", alpha=0.85, edgecolor="0.8"))
for tt in change_times:
    axs[0].axvline(tt, color="0.3", alpha=0.15, linewidth=1)

# 2) Servo commands (deg)
axs[1].step(df["t_s"], df["cmd_servo_xp_deg"], where="post", label="cmd servo_xp (deg)")
axs[1].step(df["t_s"], df["cmd_servo_xn_deg"], where="post", label="cmd servo_xn (deg)")
axs[1].step(df["t_s"], df["cmd_servo_yp_deg"], where="post", label="cmd servo_yp (deg)")
axs[1].step(df["t_s"], df["cmd_servo_yn_deg"], where="post", label="cmd servo_yn (deg)")
axs[1].set_ylabel("Cmd (deg)")
axs[1].set_title("Servo command schedule")
axs[1].grid(True)
axs[1].legend(loc="best", ncol=2)
for tt in change_times:
    axs[1].axvline(tt, color="0.3", alpha=0.10, linewidth=1)

# 3) IMU angles (decoded)
axs[2].plot(df["t_s"], df["roll"], label="roll")
axs[2].plot(df["t_s"], df["pitch"], label="pitch")
axs[2].plot(df["t_s"], df["yaw"], label="yaw")
axs[2].set_ylabel("Angle (decoded)")
axs[2].set_title("IMU angles (decoded)")
axs[2].grid(True)
axs[2].legend(loc="best")

# 4) omega_body = [p,q,r]
axs[3].plot(df["t_s"], df["p"], label="p")
axs[3].plot(df["t_s"], df["q"], label="q")
axs[3].plot(df["t_s"], df["r"], label="r")
axs[3].set_ylabel("omega_body")
axs[3].set_title("Body rates (from gyro): omega_body=[p,q,r]")
axs[3].grid(True)
axs[3].legend(loc="best")

# 5) omega_dot
axs[4].plot(df["t_s"], df["p_dot"], label="p_dot")
axs[4].plot(df["t_s"], df["q_dot"], label="q_dot")
axs[4].plot(df["t_s"], df["r_dot"], label="r_dot")
axs[4].set_ylabel("omega_dot")
axs[4].set_title("Angular acceleration: omega_dot=d(omega_body)/dt")
axs[4].grid(True)
axs[4].legend(loc="best")

axs[4].set_xlabel(t_label)
plt.tight_layout()
plt.show()

# EDF tracking error
err_df = df.dropna(subset=["cmd_edf_percent"]).copy()
err_df["edf_err"] = err_df["edf_percent"] - err_df["cmd_edf_percent"]
plt.figure(figsize=(14, 3.5))
plt.plot(err_df["t_s"], err_df["edf_err"], color="tab:red")
plt.title("EDF tracking error: telemetry - command")
plt.xlabel(t_label)
plt.ylabel("EDF error (%)")
plt.grid(True)
plt.show()
